In [1]:
import re
import time
import requests
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup as bs
from io import StringIO

In [2]:
# 1995년 1월 3일부터 오늘 날짜까지 생성
today = datetime.today().date()
start_date = datetime(1995, 1, 3).date()

dates = pd.date_range(start=today, end=start_date, freq='-1D')
date_list = [d.date() for d in dates]

print(date_list[:5])   # 최근 5일 출력
print(len(date_list))  # 총 날짜 수

[datetime.date(2025, 10, 23), datetime.date(2025, 10, 22), datetime.date(2025, 10, 21), datetime.date(2025, 10, 20), datetime.date(2025, 10, 19)]
11252


In [3]:
f"{date_list[0]}"

'2025-10-23'

In [4]:
for date in date_list[:3]:
    print(f"{date}")
    print(f"{date}".replace("-", ""))
    

2025-10-23
20251023
2025-10-22
20251022
2025-10-21
20251021


In [5]:
len(date_list)

11252

In [39]:
def new_col(final_result):
    new_col = []
    for col in final_result.columns:
        if "" in col:
            new_col.append(col[0])
        elif col[0] == col[1] == col[2]:
            new_col.append(col[0].replace(" ", "_"))
        elif col[0] != col[1] != col[2] or col[0] != (col[1] == col[2]):
            new_col.append("_".join(col).replace(" ", "_"))
    return new_col

In [40]:
result = []
for idx, date in enumerate(date_list[:2]):
    print(f"{idx}/{len(date_list[:1000])} 수집중", end="\r")
    url = "https://www.kebhana.com/cms/rate/wpfxd651_01i_01.do"
    payload = dict(ajax="true", tmpInqStrDt=f"{date}", pbldDvCd=3, inqStrDt=f"{date}".replace("-", ""), inqKindCd=1, requestTarget="searchContentDiv")
    r = requests.post(url, data=payload)
#     print(r.url)
#     print(r.status_code)
    # soup = bs(r.content, 'lxml')

    df = pd.read_html(StringIO(r.text))[0]
    df.insert(0, '날짜', f"{date}")
    result.append(df)
    time.sleep(0.2)
final_result = pd.concat(result)
final_result.columns = new_col(final_result)
final_result

,날짜,통화,현찰_사실_때_환율,현찰_사실_때_Spread,현찰_파실_때_환율,현찰_파실_때_Spread,송금_보낼_때_보낼_때,송금_받을_때_받을_때,외화_수표_파실때,매매_기준율,환가_료율,미화_환산율
0,2025-10-23,미국 USD,1457.87,1.75,1407.73,1.75,1446.80,1418.80,1416.47,1432.80,5.86524,1.0000
1,2025-10-23,일본 JPY (100),959.47,1.75,926.47,1.75,952.21,933.73,933.13,942.97,2.58682,0.6581
2,2025-10-23,유로 EUR,1696.36,1.99,1630.18,1.99,1679.90,1646.64,1644.85,1663.27,3.87900,1.1609
3,2025-10-23,중국 CNY,211.13,5.00,191.03,5.00,203.09,199.07,0.00,201.08,3.68591,0.1403
4,2025-10-23,홍콩 HKD,188.00,1.97,180.74,1.97,186.21,182.53,182.29,184.37,5.48866,0.1287
...,...,...,...,...,...,...,...,...,...,...,...,...
53,2025-10-22,리비아 LYD,0.00,0.00,0.00,0.00,266.58,260.26,0.00,263.42,2.97500,0.1838
54,2025-10-22,루마니아 RON,0.00,0.00,0.00,0.00,330.90,323.70,0.00,327.30,8.07500,0.2284
55,2025-10-22,미얀마 MMK,0.00,0.00,0.00,0.00,0.70,0.66,0.00,0.68,1.97500,0.0005
56,2025-10-22,에티오피아 ETB,0.00,0.00,0.00,0.00,9.57,9.35,0.00,9.46,2.97500,0.0066


In [41]:
final_result.columns

Index(['날짜', '통화', '현찰_사실_때_환율', '현찰_사실_때_Spread', '현찰_파실_때_환율',
       '현찰_파실_때_Spread', '송금_보낼_때_보낼_때', '송금_받을_때_받을_때', '외화_수표_파실때', '매매_기준율',
       '환가_료율', '미화_환산율'],
      dtype='object')

# 1일 전 데이터 수집하기

In [9]:
import re
import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup as bs
from io import StringIO
from sqlalchemy import create_engine, text
import pymysql
pymysql.install_as_MySQLdb()
from dbio import to_db, db_connect

In [10]:
def new_col(final_result):
    new_col = []
    for col in final_result.columns:
        if "" in col:
            new_col.append(col[0])
        elif col[0] == col[1] == col[2]:
            new_col.append(col[0].replace(" ", "_"))
        elif col[0] != col[1] != col[2] or col[0] != (col[1] == col[2]):
            new_col.append("_".join(col).replace(" ", "_"))
    return new_col

In [12]:
# 오늘부터 하루 전 날짜 생성
yesterday = datetime.today() - timedelta(days=1)
date1 = (f"{yesterday.date()}")
date2 = (f"{yesterday.date()}".replace("-", ""))

# 환율 데이터 수집
url = "https://www.kebhana.com/cms/rate/wpfxd651_01i_01.do"
payload = dict(ajax="true", tmpInqStrDt=date1, pbldDvCd=3, inqStrDt=date2, inqKindCd=1, requestTarget="searchContentDiv")
r = requests.post(url, data=payload)
df = pd.read_html(StringIO(r.text))[0]
df.insert(0, '날짜', date1)
df.columns = new_col(df)

# DB에 있는지 확인
conn = db_connect("exchage_rate")
try:
    conn.execute(text(f"select * from exchage_rate where `날짜` = {date1}"))
    print(f"{date1} 환율 정보가 이미 DB에 있습니다.")
    conn.close()
except:
    # DB에 저장
    print(f"{date1} 환율 정보가 DB에 없으므로 수집합니다.")
    to_db("exchage_rate", "exchage_rate", df)

exchage_rate 데이터베이스 확인/생성 완료
2025-10-22 환율 정보가 이미 DB에 있습니다.


In [8]:
conn.close()